# Evaluation, Robustness, and Fairness[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/parisar/BME6938-Advanced-Biomedical-AI-Spring-2027/blob/main/lectures/notebooks/week12_evaluation.ipynb)*Click the badge to open this notebook in Google Colab, then choose File > Save a copy in Drive so your changes are kept.***BME 6938 Advanced Biomedical AI, Week 12, reference notebook (ungraded)**This notebook follows the Week 12 lecture and its three clips. It is about the three ways a good-looking number can bewrong: it can be noise, it can come from the wrong signal, and it can be an average over groups the model treats verydifferently.| Section | Clip | What it builds ||---|---|---|| 1 | Clip 2, the bootstrap | bootstrap and paired bootstrap on the same 200-patient table as the clip || 2 | Clip 3, shortcuts | a model that learns a hospital marker, and three checks that catch it || 3 | Clip 3, subgroups | subgroup sensitivity with intervals, and why the small group is the uncertain one || 4 | all three | what each toy corresponds to in Goh et al. and DeGrave et al. |All data here is synthetic. Every number printed below came from running this notebook on a CPU, and the text reportswhat came out, including the places where the toy behaves differently from the paper.**Runtime.** Under a minute on a Colab CPU. No GPU, no downloads, no data files.

## 0. Setup

In [ ]:
import time                                                  # for timing the training loopimport numpy as np                                           # arraysimport torch                                                 # tensors and autogradimport torch.nn as nn                                        # layersimport matplotlib.pyplot as plt                              # figuresfrom sklearn.linear_model import LogisticRegression          # used only as a probe, in section 2torch.manual_seed(0)                                         # fixed seeds, so the printed numbers are reproducibletorch.set_num_threads(2)                                     # keep CPU usage modest on a shared machineprint("torch", torch.__version__, "| running on CPU")

## 1. How sure are we? (clip 2, "How sure are we? The bootstrap")### 1.1 The test setClip 2 used a synthetic test set of 200 patients on which model A is right 170 times (85%) and model B 164 times (82%).A difference of 3 percentage points sounds like a result. The question is how much of it is just which 200 patientshappened to be in the test set.Because both models were graded on the *same* patients, the data is not two independent accuracies: it is a 2 by 2agreement table. The clip used this one, and so does the notebook. It is illustrative, chosen to make the arithmeticcome out at 170 and 164.

In [ ]:
N = 200                                                      # patients in the test setBOTH_RIGHT, A_ONLY, B_ONLY = 156, 14, 8                      # the agreement table (synthetic, illustrative)BOTH_WRONG = N - BOTH_RIGHT - A_ONLY - B_ONLY                # whatever is left: 22# one row per patient, 1 if that model got that patient rightA = np.array([1] * BOTH_RIGHT + [1] * A_ONLY + [0] * B_ONLY + [0] * BOTH_WRONG)B = np.array([1] * BOTH_RIGHT + [0] * A_ONLY + [1] * B_ONLY + [0] * BOTH_WRONG)print(f"{'both right':>14} {BOTH_RIGHT:>4}")print(f"{'only A right':>14} {A_ONLY:>4}")print(f"{'only B right':>14} {B_ONLY:>4}")print(f"{'both wrong':>14} {BOTH_WRONG:>4}")print()print("model A:", A.sum(), "of", N, "=", f"{A.mean():.2f}")print("model B:", B.sum(), "of", N, "=", f"{B.mean():.2f}")print("difference:", f"{A.mean() - B.mean():.2f}", "which is the", A_ONLY, "-", B_ONLY, "=", A_ONLY - B_ONLY, "patients they disagree on, over", N)

### 1.2 The bootstrapResampling is the whole method: draw 200 patients from the test set *with replacement*, so some appear more than onceand some not at all, recompute the accuracy, and repeat. The spread of those recomputed accuracies estimates how muchthe accuracy would move if we had tested a different sample of 200 patients from the same population.One detail does all the work in section 1.4: each resample draws a set of **patient indices**, and both models are thenscored on that same set. That is what makes the comparison paired.

In [ ]:
R = 10000                                                    # number of bootstrap resamplesrng = np.random.default_rng(0)idx = rng.integers(0, N, (R, N))                             # R resamples, each of N patient indices, with replacementboot_A = A[idx].mean(1)                                      # model A's accuracy on each resampleboot_B = B[idx].mean(1)                                      # model B's accuracy on the SAME resampled patientsboot_D = boot_A - boot_B                                     # the paired difference, resample by resampledef ci(v, lo=2.5, hi=97.5):    """A percentile confidence interval: the middle 95% of the bootstrap distribution."""    return tuple(float(x) for x in np.percentile(v, [lo, hi]))print(f"model A   0.85   95% CI {ci(boot_A)[0]:.3f} to {ci(boot_A)[1]:.3f}")print(f"model B   0.82   95% CI {ci(boot_B)[0]:.3f} to {ci(boot_B)[1]:.3f}")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 3.4))for a, v, lab, col in ((ax[0], boot_A, "model A", "#0021A5"), (ax[0], boot_B, "model B", "#0F766E")):    a.hist(v, bins=40, alpha=0.55, color=col, label=lab)ax[0].set_title("10,000 resampled accuracies"); ax[0].set_xlabel("accuracy"); ax[0].legend()ax[1].hist(boot_D, bins=40, color="#F59E0B")ax[1].axvline(0, color="#DC2626", lw=2)                      # zero, the value that mattersax[1].set_title("the paired difference, A minus B"); ax[1].set_xlabel("difference in accuracy")for a in ax: a.grid(alpha=0.3)plt.tight_layout(); plt.show()

### 1.3 Why overlapping intervals are not a testModel A's interval and model B's interval overlap. It is tempting, and wrong, to read that as "the difference could bezero, so there is nothing here". Each marginal interval answers a question about **one model on its own**: where wouldthat model's accuracy land if we retested it? Neither of them is about the difference, and the difference has its owninterval, which the next cell computes.The two are not interchangeable, and they do not even have to agree. The cell below shows how far apart they can be: itconstructs a second, hypothetical pair of models with the same 85% and 82% accuracies but near-perfect agreement, wherethe marginal intervals overlap just as much and the paired interval excludes zero comfortably.

In [ ]:
print("the real pair, with 22 disagreements:")print(f"   paired difference 0.03, 95% CI {ci(boot_D)[0]:.3f} to {ci(boot_D)[1]:.3f}")# a hypothetical pair: same accuracies, but they disagree on only the 6 patients that separate themA2 = np.array([1] * 164 + [1] * 6 + [0] * 0 + [0] * 30)B2 = np.array([1] * 164 + [0] * 6 + [1] * 0 + [0] * 30)boot_D2 = A2[idx].mean(1) - B2[idx].mean(1)print()print("a hypothetical pair with the same 0.85 and 0.82, disagreeing on only 6 patients:")print(f"   model A2 95% CI {ci(A2[idx].mean(1))[0]:.3f} to {ci(A2[idx].mean(1))[1]:.3f}   (the same marginal interval)")print(f"   paired difference 0.03, 95% CI {ci(boot_D2)[0]:.3f} to {ci(boot_D2)[1]:.3f}   (excludes 0)")print()print("Same accuracies, same overlapping marginal intervals, opposite conclusion about the difference.")

### 1.4 Pairing, and how many patients it would takeComparing the two models on independently resampled patients throws away the pairing and widens the interval, becauseit adds back variation that cancels when the same patients are used for both.The last number below is a rough answer to the question that should follow any interval containing zero: how large wouldthe test set have to be? The standard error of the paired difference falls as one over the square root of the samplesize, so if the agreement pattern stayed the same, the required size scales by the square of the ratio of standarderrors. That assumption is the weak point: a different test set would have a different agreement pattern.

In [ ]:
# the unpaired version: resample patients separately for each modelun_A = A[rng.integers(0, N, (R, N))].mean(1)un_B = B[rng.integers(0, N, (R, N))].mean(1)boot_U = un_A - un_Bprint(f"same patients for both   95% CI {ci(boot_D)[0]:.3f} to {ci(boot_D)[1]:.3f}   width {ci(boot_D)[1] - ci(boot_D)[0]:.3f}")print(f"ignoring the pairing     95% CI {ci(boot_U)[0]:.3f} to {ci(boot_U)[1]:.3f}   width {ci(boot_U)[1] - ci(boot_U)[0]:.3f}")print(f"the unpaired interval is {(ci(boot_U)[1] - ci(boot_U)[0]) / (ci(boot_D)[1] - ci(boot_D)[0]):.1f} times wider")print()se = float(boot_D.std())                                     # the bootstrap standard error of the paired differenceneeded = N * (1.96 * se / 0.03) ** 2                         # the n at which 1.96 standard errors would equal 0.03print(f"bootstrap standard error at n = {N}: {se:.4f}")print(f"to make a 3-point difference exclude 0, you would need roughly {needed:.0f} patients,")print("assuming the two models disagree in the same pattern, which a new test set would not guarantee.")

## 2. Shortcuts (clip 3, "Shortcuts and hidden subgroups")### 2.1 Three synthetic sitesDeGrave et al. found COVID-19 classifiers reading laterality markers, annotations, image borders and patientpositioning instead of the lungs. The toy version keeps the structure and drops everything else: 32 by 32 syntheticfilms with two dark lung fields, where- the **genuine signal** is a faint diffuse opacity added inside the lung fields of positive films, deliberately weak  relative to the imaging noise, and- the **shortcut** is a bright 3 by 3 marker in the top-left corner, which at some sites is stamped on positive films.Three sites, which is the design that matters:| Site | Marker | Role ||---|---|---|| A | on positives | training and internal test || C | on positives, same convention | external validation at a new site || B | never | external validation at a site with a different convention |Site C is the case DeGrave et al. warn about: a shortcut that is present in both the training domain and the externalone. Site B is the one that exposes it.

In [ ]:
SIZE = 32yy, xx = np.mgrid[0:SIZE, 0:SIZE]LUNG = ((((xx - 11) / 5.5) ** 2 + ((yy - 16) / 9.0) ** 2) < 1) | ((((xx - 20) / 5.5) ** 2 + ((yy - 16) / 9.0) ** 2) < 1)OPACITY, NOISE = 0.035, 0.16                                 # the genuine signal is weak next to the noise, on purposedef films(n, positive, marker, r):    """n synthetic films. positive: add faint diffuse lung opacity. marker: 'tl' stamps the corner marker."""    X = np.full((n, SIZE, SIZE), 0.45, np.float32)           # the body    X[:, LUNG] = 0.22                                        # the lung fields are darker    if positive:        X[:, LUNG] += OPACITY * r.uniform(0.7, 1.3, (n, 1))  # the genuine signal    X += r.normal(0, NOISE, X.shape).astype(np.float32)      # imaging noise    if marker == "tl":        X[:, 2:5, 2:5] = 0.95                                # the shortcut: a bright corner marker    return np.clip(X, 0, 1)def make_site(n, r, marker_on_positives):    """A balanced site: half positive, half negative, with the site's own marker convention."""    k = n // 2    Xp = films(k, True, "tl" if marker_on_positives else "none", r)    Xn = films(n - k, False, "none", r)    return np.concatenate([Xp, Xn]), np.concatenate([np.ones(k, int), np.zeros(n - k, int)])rA = np.random.default_rng(0)X_train, y_train = make_site(3000, rA, True)                 # site A, trainingX_a, y_a = make_site(600, rA, True)                          # site A, internal testX_b, y_b = make_site(600, np.random.default_rng(1), False)   # site B, no markerX_c, y_c = make_site(600, np.random.default_rng(2), True)    # site C, same marker conventionprint("training films", X_train.shape, "| internal test", X_a.shape, "| site B", X_b.shape, "| site C", X_c.shape)

In [ ]:
fig, ax = plt.subplots(2, 6, figsize=(12, 4.2))panels = [(X_train[y_train == 1][:3], "site A positive"), (X_train[y_train == 0][:3], "site A negative")]for row, (imgs, label) in enumerate(panels):    for c in range(3):        ax[row, c].imshow(imgs[c], cmap="gray", vmin=0, vmax=1); ax[row, c].axis("off")    ax[row, 0].set_title(label, loc="left", fontsize=10)for row, (imgs, label) in enumerate([(X_b[y_b == 1][:3], "site B positive"), (X_c[y_c == 1][:3], "site C positive")]):    for c in range(3):        ax[row, c + 3].imshow(imgs[c], cmap="gray", vmin=0, vmax=1); ax[row, c + 3].axis("off")    ax[row, 3].set_title(label, loc="left", fontsize=10)fig.suptitle("Synthetic films. The corner marker is obvious; the lung opacity is not.", y=1.0)plt.tight_layout(); plt.show()

### 2.2 Train, then validate externallyA small convolutional network, about 17,000 parameters. The penultimate layer has 16 units, which section 2.3 uses as"the model's features".

In [ ]:
class Net(nn.Module):    """Two convolutional blocks, a 16-unit feature layer, and a two-class head."""    def __init__(self):        super().__init__()        self.body = nn.Sequential(nn.Conv2d(1, 8, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),                                  nn.Conv2d(8, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),                                  nn.Flatten(), nn.Linear(16 * 8 * 8, 16), nn.ReLU())        self.head = nn.Linear(16, 2)    def forward(self, x, feats=False):        h = self.body(x)                                     # (batch, 16): the features        return h if feats else self.head(h)                  # or the two class scoresdef as_tensor(X):    """(n, 32, 32) numpy to (n, 1, 32, 32) float tensor."""    return torch.tensor(X, dtype=torch.float32).unsqueeze(1)def train(X, y, epochs=12, seed=0):    """Train a fresh network on one site's films."""    torch.manual_seed(seed)    net = Net(); opt = torch.optim.Adam(net.parameters(), 2e-3)    Xt, yt = as_tensor(X), torch.tensor(y)    for _ in range(epochs):        for j in torch.randperm(len(Xt)).split(128):         # minibatches of 128            loss = nn.functional.cross_entropy(net(Xt[j]), yt[j])            opt.zero_grad(); loss.backward(); opt.step()    return net@torch.no_grad()def accuracy(net, X, y):    return float((net(as_tensor(X)).argmax(1).numpy() == y).mean())t0 = time.time()net = train(X_train, y_train)print(f"trained in {time.time() - t0:.0f} s,", sum(p.numel() for p in net.parameters()), "parameters")print()print(f"site A, internal test   {accuracy(net, X_a, y_a):.3f}")print(f"site C, same marker     {accuracy(net, X_c, y_c):.3f}   <- external validation, and it passes")print(f"site B, no marker       {accuracy(net, X_b, y_b):.3f}   <- the same model, a site without the convention")

Read those three numbers in that order, because the order is the argument. The model is perfect internally. It isperfect at a *new site*, which is the evidence usually offered that a model is sound. And it is at chance at a site thatdoes not use the marker.This is DeGrave et al.'s point reduced to its skeleton: "external test set validation alone may be insufficient todetect poorly behaved models". Site C passes, and the model has learned nothing about the lungs.### 2.3 Three checks that catch itNone of these needs a new site. Each asks what the model is using rather than how well it scores.

In [ ]:
# check 1: paint out the marker region and re-score the site that passedX_c_occluded = X_c.copy()X_c_occluded[:, 2:5, 2:5] = 0.45                             # replace the marker with plain body intensityprint(f"check 1, occlusion:      site C {accuracy(net, X_c, y_c):.3f} -> {accuracy(net, X_c_occluded, y_c):.3f} with the marker painted out")# check 2: a counterfactual, stamp the marker onto films the model calls negativenegatives = X_c[y_c == 0].copy()with_marker = negatives.copy(); with_marker[:, 2:5, 2:5] = 0.95with torch.no_grad():    p_before = float(torch.softmax(net(as_tensor(negatives)), 1)[:, 1].mean())    p_after = float(torch.softmax(net(as_tensor(with_marker)), 1)[:, 1].mean())print(f"check 2, counterfactual: negative films, mean P(positive) {p_before:.3f} -> {p_after:.3f} once a marker is added")# check 3: can the site be read off the model's own features?with torch.no_grad():    feats = np.concatenate([net(as_tensor(X_a), feats=True).numpy(), net(as_tensor(X_b), feats=True).numpy()])site_label = np.concatenate([np.zeros(len(X_a), int), np.ones(len(X_b), int)])perm = np.random.default_rng(5).permutation(len(feats))feats, site_label = feats[perm], site_label[perm]probe = LogisticRegression(max_iter=3000).fit(feats[:800], site_label[:800])print(f"check 3, site probe:     site predicted from the model's features at {probe.score(feats[800:], site_label[800:]):.3f} (0.50 is chance)")

### 2.4 The signal was there all alongThe most useful comparison is the last one: train the same architecture on a site that never used the marker, so thereis no shortcut to take. If that model does well, then the genuine signal was learnable and the first model simplypreferred the easier route.

In [ ]:
X_train2, y_train2 = make_site(3000, np.random.default_rng(9), False)   # same films, no markers anywherenet2 = train(X_train2, y_train2, seed=1)print(f"{'':<26}{'site A':>9}{'site B':>9}{'site C':>9}")for name, m in (("trained with the marker", net), ("trained without it", net2)):    print(f"{name:<26}{accuracy(m, X_a, y_a):>9.3f}{accuracy(m, X_b, y_b):>9.3f}{accuracy(m, X_c, y_c):>9.3f}")

## 3. Subgroups (clip 3, final segment)An overall number is a weighted average, and a weighted average is dominated by the big group. Clip 3 used sensitivityof 0.85 overall made of 0.90 in a group that is 80% of patients and 0.65 in the 20% that is not. Those numbers areillustrative; the cell below builds a synthetic cohort in which those are the *true* rates, draws 400 patients from it,and bootstraps each subgroup. That adds two things the clip could not show: the small group's estimate is the leastcertain one, and in a sample this size it can miss the truth by a wide margin.

In [ ]:
N_POS = 400                                                  # patients who actually have the conditionr = np.random.default_rng(3)group = np.where(r.random(N_POS) < 0.80, "A", "B")           # 80% in group A, 20% in group Btrue_sens = {"A": 0.90, "B": 0.65}                           # the sensitivity the model has in each groupdetected = np.array([r.random() < true_sens[g] for g in group]).astype(int)def boot_ci(v, R=10000, seed=0):    """Bootstrap percentile interval for the mean of a 0/1 array."""    rr = np.random.default_rng(seed)    b = v[rr.integers(0, len(v), (R, len(v)))].mean(1)    return float(v.mean()), tuple(float(x) for x in np.percentile(b, [2.5, 97.5]))print(f"{'group':>8}{'n':>6}{'sensitivity':>13}   95% CI")for name, mask in (("overall", np.ones(N_POS, bool)), ("A", group == "A"), ("B", group == "B")):    est, (lo, hi) = boot_ci(detected[mask], seed=7)    print(f"{name:>8}{mask.sum():>6}{est:>13.3f}   {lo:.3f} to {hi:.3f}   (width {hi - lo:.3f})")print()print("the true sensitivities used to generate this cohort were A 0.90 and B 0.65")print()wA = (group == "A").mean()print(f"the overall number is just the weighted average: {wA:.2f} x {detected[group=='A'].mean():.2f}"      f" + {1-wA:.2f} x {detected[group=='B'].mean():.2f} = {detected.mean():.2f}")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.2))names, colors = ["overall", "group A", "group B"], ["#6B7280", "#0F766E", "#DC2626"]masks = [np.ones(N_POS, bool), group == "A", group == "B"]for k, (nm, msk, col) in enumerate(zip(names, masks, colors)):    est, (lo, hi) = boot_ci(detected[msk], seed=7)    ax.errorbar(k, est, yerr=[[est - lo], [hi - est]], fmt="o", ms=9, capsize=7, lw=2.5, color=col)    ax.annotate(f"n = {msk.sum()}", (k, hi), textcoords="offset points", xytext=(0, 12), ha="center", fontsize=9, color="#6B7280")ax.set_xticks(range(3)); ax.set_xticklabels(names); ax.set_ylabel("sensitivity"); ax.set_ylim(0.45, 1.0)ax.set_title("The small group has both the worse estimate and the wider interval"); ax.grid(alpha=0.3)plt.tight_layout(); plt.show()

Three things to take from the table. The gap is real and large. The group that is doing worse is also the group youknow least about, because the interval that matters most is built from the fewest patients. And look at what happened togroup B in our run: its true sensitivity is 0.65, the 74 patients who landed in it gave an estimate near 0.74, and onlythe interval, which is about 0.20 wide, keeps the truth in view. A point estimate from 74 patients would have understatedthe problem by a lot. Reporting subgroups without intervals replaces one misleading number with several.## 4. What each toy corresponds to**Section 1 and Goh et al.** The trial reported a 2-point difference with a 95% interval from −4 to 8, and a timedifference of −82 seconds with an interval from −195 to 31. Both intervals contain zero, which is the same situation asthe paired difference here: a result, not an absence of one. The study's own standalone arm is worth the same scrutiny,since its 92% came from three runs.**Section 2 and DeGrave et al.** The correspondence is close, and the difference is instructive.| This notebook | DeGrave et al. ||---|---|| a bright corner marker | laterality markers, arrows, text annotations, image borders || the marker convention differs between sites | positive and negative films came from different archives || site B, no marker: accuracy at chance | half the predictive performance lost at a new hospital system || site C, same convention: external validation passes | patient positioning transferred between Dataset I and Dataset II || painting the marker out | masking regions and generating counterfactual radiographs || stamping a marker on a negative film | a generative network adding and removing markers || probing for the site in the features | the generalization gap between internal and external data |The toy is more extreme than the paper on purpose: our shortcut is perfectly predictive and our genuine signal is weak,so the model takes the shortcut completely and lands at chance without it. Real classifiers lose about half theirperformance, not all of it, and they do use the lungs some of the time.**Section 3** is not from either paper. It is the failure the two papers do not cover: a model that is genuinely good onaverage and genuinely bad for a minority of patients.**Things to try**1. Raise `OPACITY` from 0.035 to 0.08 and retrain. At some point the genuine signal becomes easy enough that the model   stops relying on the marker. Where is that point, and what does it tell you about how to read a shortcut result?2. Put the marker on 70% of positives instead of all of them, so it is predictive but not perfectly. Does the model   still prefer it?3. In section 1, change `A_ONLY` and `B_ONLY` while keeping 170 and 164. How much does the paired interval move, and   what does that say about reporting an accuracy difference without the agreement table?4. In section 3, make group B 5% of patients instead of 20%. The overall number barely moves; what happens to the   interval on group B?